<a href="https://colab.research.google.com/github/Anana-power/frail-analysis/blob/main/frailty_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Frailty Project

##NHANES Dataset

###Download files

In [ ]:
import io #tools for managing file-like streams in memory without writing to a physical disk
import time
from dataclasses import dataclass, field
from functools import reduce as _reduce
from pathlib import Path #in standard library

import numpy as np
import pandas as pd #outside standard library

# ============================================================================
# Constants
# ============================================================================
BASE_URL = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public"
# CDC sits behind Akamai, which serves an HTML challenge (HTTP 200) to the
# default python-requests User-Agent. A browser UA passes.
REQUEST_HEADERS = {
    "User-Agent": ("Mozilla/5.0 (X11; Linux x86_64; rv:128.0) "
                   "Gecko/20100101 Firefox/128.0"),
    "Accept": "*/*",
    "Accept-Encoding": "gzip, deflate",
} #Tells computer who u are so the server doesnt reject u
# Every SAS XPORT v5 file starts with this exact 80-byte header. Validating it
# catches CDC's soft-404 (HTTP 200 + HTML body) before pd.read_sas chokes.
XPT_MAGIC = b"HEADER RECORD*******LIBRARY HEADER RECORD" #

# nhanes_data/ lives at the repo root (this module is under replication/).
DATA_DIR = Path("nhanes_data")
DATA_DIR.mkdir(exist_ok = True) #make directory
CYCLES = ["1999-2000", "2001-2002"]
CYCLE_TAG = "NHANES_CYCLE"
# Published MEC-examined counts per cycle (CDC / ICPSR documentation).
EXPECTED_MEC = {"1999-2000": 9282, "2001-2002": 10477}

In [ ]:
# ============================================================================
# Download configuration — one FileSpec per NHANES topic
# ============================================================================
@dataclass
class FileSpec:
    name_1999: str | None = None
    name_2001: str | None = None
    variables: list = field(default_factory=list)
    variables_1999: list | None = None
    variables_2001: list | None = None

    def vars_for(self, cycle: str) -> list: #function
        if cycle == "1999-2000" and self.variables_1999 is not None:
            return self.variables_1999
        if cycle == "2001-2002" and self.variables_2001 is not None:
            return self.variables_2001
        return self.variables
#return list of variables

# Full functional-difficulty battery present in 1999-2002 (items A-S). PFQ060T
# ("push/pull large objects") does not exist until 2003-04.
_PFQ_ITEMS = ["PFQ056",
              "PFQ060A", "PFQ060B", "PFQ060C", "PFQ060D", "PFQ060E",
              "PFQ060F", "PFQ060G", "PFQ060H", "PFQ060I", "PFQ060J",
              "PFQ060K", "PFQ060L", "PFQ060M", "PFQ060N", "PFQ060O",
              "PFQ060P", "PFQ060Q", "PFQ060R", "PFQ060S"]
# CIDI depression items named in Table S2. Administered to ages 20-39 only in
# 1999-2002, so all-missing for age>=45 (they become the depression placeholders).
_CIQ_DEP = ["CIQD001", "CIQD002", "CIQD008", "CIQD009", "CIQD012", "CIQD013",
            "CIQD019", "CIQD022", "CIQD025", "CIQD026", "CIQD029", "CIQD043"]

FILE_MAP: dict[str, FileSpec] = {
    # --- demographics + the predictor topics feeding the 75 candidates ---
    "demo": FileSpec("DEMO", "DEMO_B", #changed name between cycles, but variables are the same
                     variables=["RIDSTATR", "RIDAGEYR", "RIAGENDR"]),
    "bmx": FileSpec("BMX", "BMX_B", variables=["BMXHT", "BMXWT", "BMXBMI"]),
    "bpx": FileSpec("BPX", "BPX_B",
                    variables=["BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4",
                               "BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4", "BPXPLS"]),
    "pfq": FileSpec("PFQ", "PFQ_B",
                    variables_1999=_PFQ_ITEMS, variables_2001=_PFQ_ITEMS),
    "mcq": FileSpec("MCQ", "MCQ_B",
                    variables=["MCQ160A", "MCQ160B", "MCQ160C", "MCQ160D",
                               "MCQ160E", "MCQ160F", "MCQ160I", "MCQ160K",
                               "MCQ160M", "MCD160M", "MCQ160P", "MCQ220"]),
    "bpq": FileSpec("BPQ", "BPQ_B", variables=["BPQ020"]),
    "diq": FileSpec("DIQ", "DIQ_B", variables=["DIQ010"]),
    "kiq": FileSpec("KIQ", "KIQ_U_B",
                    variables_1999=["KIQ020", "KIQ040"],
                    variables_2001=["KIQ022", "KIQ050"]), #diff variables
    "huq": FileSpec("HUQ", "HUQ_B",
                    variables=["HUQ010", "HUQ020", "HUQ050", "HUQ051",
                               "HUQ070", "HUD070", "HUQ071"]),
    "rxq": FileSpec("RXQ_RX", "RXQ_RX_B", variables=["RXD030", "RXD295"]),
    "ciq_dep": FileSpec("CIQMDEP", "CIQDEP_B", variables=_CIQ_DEP),
    "cbc": FileSpec("LAB25", "L25_B",
                    variables=["LBXHGB", "LBXPLTSI", "LBXMCVSI", "LBXRDW",
                               "LBXNEPCT", "LBXLYPCT", "LBXRBCSI"]),
    "biochem": FileSpec("LAB18", "L40_B",
                        variables=["LBXSAL", "LBXSAPSI", "LBDSAPSI", "LBXSLDSI",
                                   "LBDSLDSI", "LBDSTBSI", "LBDSBUSI", "LBXSCR",
                                   "LBDSCR", "LBDSCRSI", "LBXSC3SI", "LBDSUASI",
                                   "LBDSPHSI", "LBXSNASI", "LBDSCASI", "LBXSTR",
                                   "LBDSTRSI", "LBXSGL", "LBDSGLSI", "LBXSIR",
                                   "LBDSIRSI", "LBXSTP"]),
    "lipids": FileSpec("LAB13", "L13_B",
                       variables=["LBXTC", "LBDTCSI", "LBDHDLSI", "LBXHDD",
                                  "LBDHDD"]),
    "hba1c": FileSpec("LAB10", "L10_B", variables=["LBXGH"]),
    "crp": FileSpec("LAB11", "L11_B", variables=["LBXCRP"]),
    "folate_b12": FileSpec("LAB06", "L06_B",
                           variables=["LBXRBF", "LBDRBFSI", "LBXB12", "LBDB12SI"]),
    "vit_d": FileSpec(None, "L06VID_B", variables=["LBXVID"]),
    # --- Fried-component inputs: NOT used by the predictor pipeline, downloaded
    #     so you have complete data for building the frailty LABEL yourself. ---
    "whq": FileSpec("WHQ", "WHQ_B", variables=["WHD050", "WHD020"]),
    "paq": FileSpec("PAQ", "PAQ_B", variables=["PAQ180"]),
    "msx": FileSpec("MSX", "MSX_B",
                    variables_1999=["MSXWTIME", "MSAEXSTS", "MSAWDEV"],
                    variables_2001=["MSXW20TM", "MSAEXSTS", "MSAWDEV"]),
}


# ============================================================================
# The 75 candidate predictors (Table S2). (domain, label, [NHANES codes]).
# Codes are coalesced row-wise (first non-null wins) to absorb cycle-variant
# item renames. Labels are the stable column names used downstream.
# ============================================================================
S2_ROWS = [
    ("Cognition", "confusion/memory", ["PFQ056", "PFQ057"]), #cat, name, code
    ("FuncDep", "managing money", ["PFQ060A"]),
    ("FuncDep", "walk quarter mile", ["PFQ060B"]),
    ("FuncDep", "walk up 10 steps", ["PFQ060C"]),
    ("FuncDep", "stoop/crouch/kneel", ["PFQ060D"]),
    ("FuncDep", "lifting/carrying", ["PFQ060E"]),
    ("FuncDep", "house chores", ["PFQ060F"]),
    ("FuncDep", "preparing meals", ["PFQ060G"]),
    ("FuncDep", "stand from chair", ["PFQ060I"]),
    ("FuncDep", "in/out of bed", ["PFQ060J"]),
    ("FuncDep", "fork/knife/cup", ["PFQ060K"]),
    ("FuncDep", "dressing", ["PFQ060L"]),
    ("FuncDep", "stand long periods", ["PFQ060M"]),
    ("FuncDep", "grasp small objects", ["PFQ060P"]),
    ("FuncDep", "attend social event", ["PFQ060R"]),
    ("FuncDep", "leisure at home", ["PFQ060S"]),
    ("FuncDep", "push/pull large [NA]", ["PFQ060T", "PFQ061T"]),
    ("Depress", "little interest [NA]", ["CIQD008", "CIQD009", "DPQ010"]),
    ("Depress", "feeling down [NA]", ["DPQ020", "CIQD001", "CIQD002"]),
    ("Depress", "trouble sleeping [NA]", ["DPQ030", "CIQD025", "CIQD026"]),
    ("Depress", "feeling tired [NA]", ["DPQ040", "CIQD012", "CIQD013"]),
    ("Depress", "poor appetite [NA]", ["CIQD019", "CIQD022", "DPQ050"]),
    ("Depress", "feeling bad self [NA]", ["DPQ060", "CIQD029"]),
    ("Depress", "trouble concentr [NA]", ["DPQ070", "CIQD043"]),
    ("Comorb", "arthritis", ["MCQ160A"]), #keep it
    ("Comorb", "thyroid", ["MCQ160I", "MCD160M", "MCQ160M"]),
    ("Comorb", "chronic bronchitis", ["MCQ160K", "MCQ160P"]),
    ("Comorb", "cancer", ["MCQ220"]),
    ("Comorb", "CHF", ["MCQ160B"]),
    ("Comorb", "coronary hd", ["MCQ160C"]),
    ("Comorb", "angina", ["MCQ160D"]),
    ("Comorb", "heart attack", ["MCQ160E"]), #scale
    ("Comorb", "stroke", ["MCQ160F"]),
    ("Comorb", "high blood pressure", ["BPQ020"]), #scale
    ("Comorb", "diabetes", ["DIQ010"]),
    ("Comorb", "weak/failing kidney", ["KIQ020", "KIQ022"]),
    ("Comorb", "urine leakage", ["KIQ040", "KIQ050"]),
    ("HealthUtil", "general health", ["HUQ010"]),
    ("HealthUtil", "health vs 1yr ago", ["HUQ020"]),
    ("HealthUtil", "overnight hospital", ["HUQ070", "HUD070", "HUQ071"]),
    ("HealthUtil", "times healthcare", ["HUQ050", "HUQ051"]),
    ("HealthUtil", "num prescriptions", ["RXD030", "RXD295"]),
    ("Anthro", "BMI (ht,wt)", ["BMXHT", "BMXWT", "BMXBMI"]),
    ("BP/Pulse", "diastolic", ["BPXDI1"]), #scale
    ("BP/Pulse", "systolic", ["BPXSY1"]), #scale
    ("BP/Pulse", "MAP (derived)", ["BPXSY1"]),
    ("BP/Pulse", "pulse pressure (derived)", ["BPXSY1"]), #scale
    ("BP/Pulse", "pulse", ["BPXPLS"]), #scale
    ("Hemato", "hemoglobin", ["LBXHGB"]), #scale
    ("Hemato", "platelet", ["LBXPLTSI"]),
    ("Hemato", "mean cell volume", ["LBXMCVSI"]),
    ("Hemato", "RDW", ["LBXRDW"]),
    ("Hemato", "seg neutrophils %", ["LBXNEPCT"]),
    ("Liver", "RBC count", ["LBXRBCSI"]),
    ("Liver", "albumin", ["LBXSAL", "LBDSALSI"]), #scale
    ("Liver", "alkaline phosphatase", ["LBXSAPSI", "LBDSAPSI"]), #scale
    ("Liver", "bilirubin total", ["LBDSTBSI"]), #scale
    ("Liver", "LDH", ["LBXSLDSI", "LBDSLDSI"]),
    ("Renal", "BUN", ["LBDSBUSI"]),
    ("Renal", "creatinine", ["LBDSCRSI"]), #scale
    ("Renal", "bicarbonate", ["LBXSC3SI"]), #scale
    ("Renal", "uric acid", ["LBDSUASI"]), #scale
    ("Renal", "phosphorus", ["LBDSPHSI"]), #scale
    ("Renal", "sodium", ["LBXSNASI"]), #scale
    ("Renal", "calcium total", ["LBDSCASI"]), #scale
    ("Cardio", "HDL", ["LBDHDLSI", "LBDHDD", "LBXHDD"]), #scale
    ("Cardio", "total cholesterol", ["LBDTCSI", "LBXTC"]), #scale
    ("Cardio", "triglyceride", ["LBDSTRSI"]), #scale
    ("Cardio", "glucose serum", ["LBDSGLSI"]), #scale
    ("Cardio", "glycohemoglobin", ["LBXGH"]), #scale
    ("Cardio", "CRP", ["LBXCRP"]), #scale
    ("Trace", "iron", ["LBDSIRSI"]), #scale
    ("Trace", "RBC folate", ["LBDRBFSI", "LBXRBF"]), #scale
    ("Trace", "vitamin B12", ["LBDB12SI", "LBXB12"]), #scale
    ("Trace", "vitamin D", ["LBXVID"]), #scale
]
assert len(S2_ROWS) == 75, f"S2_ROWS has {len(S2_ROWS)}, expected 75"

_LABEL_DOMAIN: dict[str, str] = {label: dom for dom, label, _ in S2_ROWS}

# The 8 predictors unavailable in NHANES 1999-2002 (all-NaN placeholders): the
# 2003+ push/pull item and the 7 depressive-symptom rows (CIDI is ages 20-39
# only, PHQ-9/DPQ is 2005+). These drop at the 20%-missing screen.
PLACEHOLDER_LABELS: set[str] = {
    "push/pull large [NA]", "little interest [NA]", "feeling down [NA]",
    "trouble sleeping [NA]", "feeling tired [NA]", "poor appetite [NA]",
    "feeling bad self [NA]", "trouble concentr [NA]",
} #set = unique values only

# Functional-dependence difficulty items -> encoded to ordinal 0..3.
DIFFICULTY_LABELS: list[str] = [label for dom, label, _ in S2_ROWS if dom == "FuncDep"]

# The paper's final 8 features; the 6 that live among the 75 screened candidates
# (age and sex are demographic keys carried separately, not screened here).
FINAL_8 = ["age", "sex", "bmi", "pulse_pressure", "creatinine", "hemoglobin",
           "preparing meals", "lifting/carrying"]
FINAL8_IN_CANDIDATES = ["bmi", "pulse_pressure", "creatinine", "hemoglobin",
                        "lifting/carrying", "preparing meals"]

# S2 labels renamed to canonical derived-column names in build_feature_matrix.
_DERIVED_RENAME = {
    "BMI (ht,wt)": "bmi",
    "diastolic": "mean_dbp",
    "systolic": "mean_sbp",
    "MAP (derived)": "map",
    "pulse pressure (derived)": "pulse_pressure",
} #dict
# HealthUtil is heterogeneous: 'overnight hospital' is a yes/no item; 'general
# health' and 'health vs 1yr ago' are ordinal scales (null 7/9); the remaining
# two are genuine counts (7/9 are valid) and pass through as labs.
_HEALTHUTIL_YESNO = {"overnight hospital"} #set
_HEALTHUTIL_ORDINAL = {"general health", "health vs 1yr ago"} #set

In [ ]:
# ============================================================================
# Download helpers
# ============================================================================
def _cycle_url(cycle: str, base_name: str) -> str: #underscore = used internally, specific to this code
    first_year = cycle.split("-")[0]
    return f"{BASE_URL}/{first_year}/DataFiles/{base_name}.XPT"


def _fetch_xpt(url, cache_dir, retries=3, backoff=2.0):
    """Download one XPT file -> DataFrame (cached). None if the file 404s."""
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = cache_dir / url.split("/")[-1]
    if cache_path.exists():
        return pd.read_sas(cache_path, format="xport")

    import requests  # lazy: cached/offline runs never need the network
    last_err = None
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=REQUEST_HEADERS, timeout=120,
                             allow_redirects=True)
            if r.status_code == 404:
                return None
            r.raise_for_status()
            content = r.content
            if not content.startswith(XPT_MAGIC):
                raise ValueError(
                    f"response not an XPT file (soft-404?): status={r.status_code}"
                    f" content-type={r.headers.get('Content-Type', '?')!r}")
            cache_path.write_bytes(content)
            return pd.read_sas(io.BytesIO(content), format="xport")
        except Exception as e:  # noqa: BLE001
            last_err = e
            time.sleep(backoff * (attempt + 1))
    raise RuntimeError(f"failed to fetch {url}: {last_err}")


def _load_cycle(cycle, cache_root, verbose):
    cache_dir = Path(cache_root) / cycle.split("-")[0]
    tables = {}
    for topic, spec in FILE_MAP.items():
        name = spec.name_1999 if cycle == "1999-2000" else spec.name_2001
        if name is None:
            continue
        df = _fetch_xpt(_cycle_url(cycle, name), cache_dir)
        if df is None:
            continue
        df.columns = [c.upper() for c in df.columns]
        keep = ["SEQN"] + [v for v in spec.vars_for(cycle) if v in df.columns]
        tables[topic] = df[keep].copy()
        if verbose:
            print(f"  [{cycle}] {topic}: {df.shape[0]} rows")
    return tables


def _merge_cycle(tables):
    """Outer-merge per-topic tables on SEQN; collapse RX to per-person first."""
    tables = dict(tables)
    if "rxq" in tables and "RXD295" in tables["rxq"].columns:
        tables["rxq"] = tables["rxq"].groupby("SEQN", as_index=False)[
            "RXD295"].max()
    return _reduce(lambda l, r: pd.merge(l, r, on="SEQN", how="outer"),
                   tables.values())

###Download Raw

In [ ]:
# ============================================================================
# Stage 0 — download_raw()
# ============================================================================
def download_raw(cache_root=DATA_DIR, verbose=False) -> pd.DataFrame:
    """Download and merge every NHANES topic for both cycles into one raw panel.

    Returns one row per SEQN, restricted to MEC-examined participants
    (RIDSTATR == 2) — the paper's 19,759 initial sample — with RAW, unencoded
    NHANES values and no derived columns. This is the immutable panel every
    later stage reads and never mutates. Files cache under ``cache_root`` so
    re-runs are offline.
    """
    panels = []
    for cycle in CYCLES: #all caps = global variable (exists everywhere)
        if verbose: #unecessarily wordy
            print(f"[download] cycle {cycle} ...")
        panel = _merge_cycle(_load_cycle(cycle, cache_root, verbose))
        panel[CYCLE_TAG] = cycle
        panels.append(panel)
    panel = pd.concat(panels, ignore_index=True, sort=False)

    if "RIDSTATR" not in panel.columns:
        raise KeyError("RIDSTATR absent — cannot identify the MEC sample.")
    raw = panel[panel["RIDSTATR"] == 2].copy().reset_index(drop=True) #subselecting the condition in the [] #reset numbering

    # pd.read_sas decodes the SAS special-missing sentinel as a denormalized
    # float (~5.4e-79) — neither NaN nor 0 — which would silently corrupt
    # derived features. Null everything of denormal magnitude, once, here.
    num_cols = raw.select_dtypes(include=[np.number]).columns
    raw[num_cols] = raw[num_cols].mask(raw[num_cols].abs() < 1e-30)
    return raw #turning numbers smaller than 1e-30 to 0


# ============================================================================
# Shared feature helpers (also useful standalone)
# ============================================================================
def _bp_mean(frame, prefix):
    """Row-wise mean of BP readings 1-4, with the 0-as-missing convention."""
    cols = [f"{prefix}{i}" for i in (1, 2, 3, 4) if f"{prefix}{i}" in frame.columns]
    if not cols:
        return pd.Series(np.nan, index=frame.index)
    return frame[cols].replace(0, np.nan).mean(axis=1)


def pulse_pressure(frame) -> pd.Series:
    """Pulse pressure (mean systolic - mean diastolic), aligned to ``frame``."""
    return _bp_mean(frame, "BPXSY") - _bp_mean(frame, "BPXDI")


def difficulty_ordinal(series) -> pd.Series:
    """PFQ060x difficulty -> ordinal 0..3 (0 none, 1 some, 2 much, 3 unable).
    Refused (7) / don't-know (9) -> 0. Blanks stay NaN.

    Emits INTEGER codes, never the strings None/Low/Med/High: 'None' is one of
    pandas' default NA tokens, so a 'None' label would round-trip to missing.
    """
    return series.map({1: 0, 2: 1, 3: 2, 4: 3, 7: 0, 9: 0})

###Select cohort

In [ ]:
# ============================================================================
# Stage 1 — select_cohort()
# ============================================================================
def cohort_mask(raw) -> pd.Series:
    """Boolean mask of the analytic cohort within ``raw`` (see select_cohort)."""
    return (raw["PFQ060E"].isin([1, 2, 3, 4])
            & (raw["RIDAGEYR"] >= 45)
            & pulse_pressure(raw).notna())
# 3 requirements

def select_cohort(raw) -> pd.DataFrame:
    """Select the analytic cohort: the closest public reconstruction of the
    paper's n = 3,480 training sample.

    Rule: age >= 45, weakness measured (PFQ060E lifting/carrying answered), and
    a valid pulse pressure (blood-pressure exam completed). Yields n = 3,487 and
    reproduces 8 of the paper's 9 Table S3 baseline metrics to within ~2% (all
    but mean age). See REPLICATION.md for why the exact 3,480 is not reachable
    on public NHANES.
    """
    return raw[cohort_mask(raw)].copy()
#select only the ones who fit the requirements --> final amount n = 3487

def fingerprint(participants) -> dict:
    """The Table S3 baseline 'fingerprint' for a set of participants: the 8
    baseline metrics used to judge a cohort against the paper."""
    lift = difficulty_ordinal(participants["PFQ060E"]).dropna()
    meal = difficulty_ordinal(participants["PFQ060G"]).dropna()
    return {
        "n": len(participants),
        "age": participants["RIDAGEYR"].mean(),
        "male_pct": (participants["RIAGENDR"] == 1).mean() * 100,
        "bmi": participants["BMXBMI"].mean(),
        "pulse_pressure": pulse_pressure(participants).mean(),
        "creatinine": participants["LBDSCRSI"].mean(),
        "hemoglobin": participants["LBXHGB"].mean(),
        "lifting_none": (lift == 0).mean() * 100 if len(lift) else np.nan,
        "meals_none": (meal == 0).mean() * 100 if len(meal) else np.nan,
    }


# ============================================================================
# Stage 2/3 — build_feature_matrix()
# ============================================================================
def _classify(label):
    """Map an S2 label to its encoder type."""
    if label in _DERIVED_RENAME:
        return "derived"
    dom = _LABEL_DOMAIN.get(label)
    if dom == "FuncDep":
        return "difficulty"
    if dom in ("Comorb", "Cognition") or label in _HEALTHUTIL_YESNO:
        return "condition"
    if label in _HEALTHUTIL_ORDINAL:
        return "ordinal"
    return "lab"

def build_feature_matrix(participants, return_provenance=False):
    """Build the encoded 75-column candidate matrix for a set of participants.

    Resolves each of the 75 Table S2 predictors against the participants
    (coalescing cycle-variant codes), then encodes per type:
      * difficulty -> ordinal 0..3 (structural skips filled to 0, per NHANES);
      * condition  -> 0/1 (yes/no items; refused/don't-know -> NaN);
      * ordinal    -> HealthUtil scales, refused/don't-know (7/9) nulled;
      * lab        -> pass through unchanged;
      * derived    -> bmi, map, pulse_pressure, mean_sbp, mean_dbp computed from
                      blood-pressure readings (0-as-missing) and BMI.
    The 8 unavailable predictors (PLACEHOLDER_LABELS) are left all-NaN so they
    drop at the missingness screen. ``participants`` is never mutated.

    Returns the matrix, or (matrix, provenance) if ``return_provenance``.
    """
    sub = participants
    present = set(sub.columns)
    matrix = pd.DataFrame(index=sub.index)
    prov = []
    for dom, label, codes in S2_ROWS:
        avail = [c for c in codes if c in present]
        col = None
        for c in avail:
            col = sub[c].copy() if col is None else col.fillna(sub[c])
        if col is None or col.notna().sum() == 0:
            matrix[label] = np.nan
            prov.append((dom, label, "placeholder"))
        else:
            matrix[label] = col.values
            prov.append((dom, label, "available"))

    # Encode each resolved (non-placeholder, non-derived) column by type.
    placeholders = {c for c in matrix.columns if matrix[c].isna().all()}
    for label in matrix.columns:
        if label in placeholders or label in _DERIVED_RENAME:
            continue
        ctype = _classify(label)
        if ctype == "difficulty":
            # Structural skips (not-limited screen-outs) fill to 0 (no difficulty).
            matrix[label] = difficulty_ordinal(matrix[label]).fillna(0)
        elif ctype == "condition":
            matrix[label] = matrix[label].map(
                lambda v: 1.0 if v == 1 else (0.0 if v == 2 else np.nan))
        elif ctype == "ordinal":
            matrix[label] = matrix[label].mask(matrix[label].isin([7, 9]))
        # lab -> pass through

    # Derived clinical columns, computed from the participants' BP readings.
    matrix = matrix.rename(columns=_DERIVED_RENAME)
    mean_sbp, mean_dbp = _bp_mean(sub, "BPXSY"), _bp_mean(sub, "BPXDI")
    matrix["mean_sbp"] = mean_sbp.values
    matrix["mean_dbp"] = mean_dbp.values
    matrix["bmi"] = sub["BMXBMI"].values
    matrix["map"] = (mean_dbp + (mean_sbp - mean_dbp) / 3).values
    matrix["pulse_pressure"] = (mean_sbp - mean_dbp).values

    if return_provenance:
        prov_df = pd.DataFrame(prov, columns=["domain", "label", "status"])
        return matrix, prov_df
    return matrix
    #unknown values = NaN

###Reduce unsupervised


In [ ]:
# ============================================================================
# Stage 4 — reduce_unsupervised()
# ============================================================================
# map and pulse_pressure are exact linear combinations of mean_sbp/mean_dbp, so
# their high correlation with those sources is an algebraic identity, not
# discovered redundancy. Exempting these intra-family pairs from the collinearity
# screen keeps the clinically-meaningful derived features (the paper's final-8
# pulse_pressure) from being eliminated by construction.
_BP_DERIVED = {"map", "pulse_pressure"}
_BP_FAMILY = _BP_DERIVED | {"mean_sbp", "mean_dbp"}


def _is_derived_bp_pair(a, b):
    return a in _BP_FAMILY and b in _BP_FAMILY and (
        a in _BP_DERIVED or b in _BP_DERIVED)


def reduce_unsupervised(matrix, missing_thresh=0.20, drop_missing=True,
                        drop_nzv=True, nzv_dominance=0.99, drop_collinear=True,
                        collinear_thresh=0.7, exempt_derived_bp=True):
    """The paper's unsupervised feature screen, as three independent sub-screens
    over whatever matrix you pass:

      (a) missing     — drop columns with > ``missing_thresh`` fraction missing;
      (b) nzv         — drop near-zero-variance columns (single value's share
                        exceeds ``nzv_dominance``, or constant);
      (c) collinear   — for each pair with |Pearson r| > ``collinear_thresh``,
                        drop the later column. When ``exempt_derived_bp`` (the
                        default), BP-family pairs involving a derived member are
                        skipped (their collinearity is an identity).

    Returns (reduced, report). ``report`` names each dropped column, the screen
    that dropped it, and the triggering value. ``matrix`` is never mutated.

    Column count depends on the input population (see the module docstring):
    on the analytic cohort at the default r > 0.7 this yields 61; the paper's 60
    is reproduced on the age>=45 universe at collinear_thresh=0.72.
    """
    cur = matrix.copy()
    report = []

    if drop_missing:
        for c in list(cur.columns):
            frac = float(cur[c].isna().mean())
            if frac > missing_thresh:
                report.append((c, "missing", round(frac, 4)))
                cur = cur.drop(columns=c)
#drops the column if the amount of missing values exceed the threshold

    if drop_nzv:
        for c in list(cur.columns):
            s = cur[c].dropna()
            if len(s) == 0 or s.nunique() <= 1:
                report.append((c, "nzv", "constant"))
                cur = cur.drop(columns=c)
                continue
            top_share = float(s.value_counts().iloc[0]) / len(s)
            if top_share > nzv_dominance:
                report.append((c, "nzv", round(top_share, 4)))
                cur = cur.drop(columns=c)
#drop columns that have values near or = 0
    if drop_collinear:
        corr = cur.select_dtypes(include=[np.number]).corr().abs()
        cols = list(corr.columns)
        removed = set()
        for i, ci in enumerate(cols):
            if ci in removed:
                continue
            for cj in cols[i + 1:]:
                if cj in removed:
                    continue
                if exempt_derived_bp and _is_derived_bp_pair(ci, cj):
                    continue
                r = corr.loc[ci, cj]
                if pd.notna(r) and r > collinear_thresh:
                    report.append((cj, "collinear", f"|r|={round(float(r), 3)} "
                                                    f"with {ci}"))
                    removed.add(cj)
        cur = cur.drop(columns=list(removed))
#drops columns that are similar to each other and keep one only
    return cur, pd.DataFrame(report, columns=["column", "screen", "value"])

In [ ]:
print("NHANES 1999-2002 frailty replication pipeline (Lai et al. 2025)")
raw = download_raw(verbose=True)
print(f"\n  raw MEC panel        {len(raw):>7,} rows")
cohort = select_cohort(raw)
print(f"  analytic cohort      {len(cohort):>7,} participants")
matrix = build_feature_matrix(cohort)
print(f"  candidate matrix     {matrix.shape[0]:>7,} x {matrix.shape[1]}")
reduced, report = reduce_unsupervised(matrix)
n = {s: int((report['screen'] == s).sum())
      for s in ('missing', 'nzv', 'collinear')}
print(f"  reduced (cohort)     {reduced.shape[0]:>7,} x {reduced.shape[1]}"
      f"   (dropped {len(report)}: {n['missing']} missing, {n['nzv']} nzv, "
      f"{n['collinear']} collinear)")
#results show that we have 11 dropped columns because of too many missing values
#0 dropped because of almost near or = 0
#3 are dropped because they are similar to others

NHANES 1999-2002 frailty replication pipeline (Lai et al. 2025)
[download] cycle 1999-2000 ...
  [1999-2000] demo: 9965 rows
  [1999-2000] bmx: 9282 rows
  [1999-2000] bpx: 9282 rows
  [1999-2000] pfq: 9493 rows
  [1999-2000] mcq: 9493 rows
  [1999-2000] bpq: 6044 rows
  [1999-2000] diq: 9493 rows
  [1999-2000] kiq: 4880 rows
  [1999-2000] huq: 9965 rows
  [1999-2000] rxq: 15643 rows
  [1999-2000] ciq_dep: 778 rows
  [1999-2000] cbc: 8832 rows
  [1999-2000] biochem: 6758 rows
  [1999-2000] lipids: 8344 rows
  [1999-2000] hba1c: 6758 rows
  [1999-2000] crp: 8344 rows
  [1999-2000] folate_b12: 8832 rows
  [1999-2000] whq: 6044 rows
  [1999-2000] paq: 9188 rows
  [1999-2000] msx: 2156 rows
[download] cycle 2001-2002 ...
  [2001-2002] demo: 11039 rows
  [2001-2002] bmx: 10477 rows
  [2001-2002] bpx: 10477 rows
  [2001-2002] pfq: 10470 rows
  [2001-2002] mcq: 10470 rows
  [2001-2002] bpq: 6634 rows
  [2001-2002] diq: 10470 rows
  [2001-2002] kiq: 5411 rows
  [2001-2002] huq: 11039 rows
  [2

In [ ]:
matrix

,confusion/memory,managing money,walk quarter mile,walk up 10 steps,stoop/crouch/kneel,lifting/carrying,house chores,preparing meals,stand from chair,in/out of bed,...,HDL,total cholesterol,triglyceride,glucose serum,glycohemoglobin,CRP,iron,RBC folate,vitamin B12,vitamin D
1,0.0,0,0.0,0.0,0,0,0,0,0,0,...,1.39,5.56,1.298,4.330,4.7,0.36,11.28,3284.3,521.77,NaN
12,0.0,0,2.0,2.0,2,1,2,0,1,0,...,1.28,8.12,3.635,7.882,7.6,0.27,12.18,1259.3,264.94,NaN
13,0.0,0,0.0,0.0,0,0,0,0,0,0,...,1.04,4.50,0.756,6.384,5.8,1.99,11.82,566.3,298.89,NaN
15,1.0,0,0.0,0.0,0,1,0,0,0,0,...,1.41,4.24,0.474,4.163,4.6,0.04,8.06,1164.2,576.38,NaN
23,0.0,0,1.0,1.0,3,1,1,0,0,0,...,2.72,5.66,0.847,4.718,5.3,0.08,17.91,634.2,479.70,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19746,0.0,0,0.0,0.0,0,0,0,0,0,0,...,1.89,6.93,1.795,4.940,5.8,0.17,18.45,627.4,434.68,7.0
19748,1.0,0,0.0,0.0,0,0,0,0,0,0,...,1.01,7.55,2.303,4.552,5.3,0.21,13.25,303.5,148.34,8.0
19749,0.0,0,0.0,0.0,0,0,0,0,0,0,...,1.42,4.58,1.106,3.719,5.9,0.03,12.36,824.5,514.39,29.0
19750,0.0,0,0.0,0.0,2,3,3,3,3,1,...,1.16,5.43,1.219,7.716,7.1,1.19,9.13,1048.7,481.18,29.0


In [ ]:
matrix.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3487 entries, 1 to 19753
Data columns (total 75 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   confusion/memory       3481 non-null   float64
 1   managing money         3487 non-null   int64  
 2   walk quarter mile      3487 non-null   float64
 3   walk up 10 steps       3487 non-null   float64
 4   stoop/crouch/kneel     3487 non-null   int64  
 5   lifting/carrying       3487 non-null   int64  
 6   house chores           3487 non-null   int64  
 7   preparing meals        3487 non-null   int64  
 8   stand from chair       3487 non-null   int64  
 9   in/out of bed          3487 non-null   int64  
 10  fork/knife/cup         3487 non-null   int64  
 11  dressing               3487 non-null   int64  
 12  stand long periods     3487 non-null   int64  
 13  grasp small objects    3487 non-null   int64  
 14  attend social event    3487 non-null   int64  
 15  leisure 

##Unsupervised filtering

In [ ]:
import pandas as pd
import numpy as np

def unsupervised_feature_filtering(df, missing_threshold=0.20, variance_threshold=0.01, corr_threshold=0.85, keep_cols=None):
    """
    Performs unsupervised feature filtering on the raw feature matrix.

    Parameters:
    -----------
    df : pd.DataFrame
        The merged dataset containing all features and target label.
    missing_threshold : float (default=0.20)
        Columns with missing proportion > missing_threshold are dropped.
    variance_threshold : float (default=0.01)
        Columns with variance below this value are dropped.
    corr_threshold : float (default=0.85)
        For pairs with Pearson correlation > corr_threshold, one redundant feature is dropped.
    keep_cols : list
        List of essential columns (e.g., 'SEQN', 'frailty_label') to protect from being dropped.

    Returns:
    --------
    filtered_df : pd.DataFrame
        DataFrame after removing low-quality, low-variance, and collinear features.
    report : dict
        Summary report of dropped features at each filtering stage.
    """
    if keep_cols is None:
        keep_cols = ['SEQN', 'frailty_label', 'frailty_status', 'age', 'sex', 'gender']

    # Standardize column name list to match protected columns present in df
    protected = [c for c in keep_cols if c in df.columns]
    candidate_features = [c for c in df.columns if c not in protected]

    dropped_report = {
        'high_missing': [],
        'low_variance': [],
        'high_correlation': []
    }

    current_df = df.copy()

    # -------------------------------------------------------------------------
    # STEP 1: Filter High Missing Rates (> 20%)
    # -------------------------------------------------------------------------
    missing_rates = current_df[candidate_features].isna().mean()
    high_missing_cols = missing_rates[missing_rates > missing_threshold].index.tolist()

    dropped_report['high_missing'] = high_missing_cols
    candidate_features = [c for c in candidate_features if c not in high_missing_cols]

    print(f"[Step 1] Dropped {len(high_missing_cols)} columns with > {missing_threshold*100:.0f}% missing values.")
#dont want columns with more than 20% of missing values

    # -------------------------------------------------------------------------
    # STEP 2: Filter Low / Near-Zero Variance
    # -------------------------------------------------------------------------
    # Temporarily fill missing values with median for variance check
    temp_numeric = current_df[candidate_features].select_dtypes(include=[np.number])
    variances = temp_numeric.var()
    low_var_cols = variances[variances <= variance_threshold].index.tolist()

    dropped_report['low_variance'] = low_var_cols
    candidate_features = [c for c in candidate_features if c not in low_var_cols]

    print(f"[Step 2] Dropped {len(low_var_cols)} columns with variance <= {variance_threshold}.")

    # -------------------------------------------------------------------------
    # STEP 3: Filter Highly Collinear Features (Pearson Correlation > 0.85)
    # -------------------------------------------------------------------------
    numeric_candidates = current_df[candidate_features].select_dtypes(include=[np.number])
    corr_matrix = numeric_candidates.corr().abs()

    # Select upper triangle of correlation matrix
    upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

    # Find features with correlation greater than threshold
    high_corr_cols = [column for column in upper_tri.columns if any(upper_tri[column] > corr_threshold)]

    dropped_report['high_correlation'] = high_corr_cols
    candidate_features = [c for c in candidate_features if c not in high_corr_cols]

    print(f"[Step 3] Dropped {len(high_corr_cols)} redundant features with Pearson |r| > {corr_threshold}.")

    # -------------------------------------------------------------------------
    # FINAL RECONSTRUCTION
    # -------------------------------------------------------------------------
    final_cols = protected + candidate_features
    filtered_df = current_df[final_cols]

    print("\n=== UNSUPERVISED FILTERING SUMMARY ===")
    print(f"Original total columns : {df.shape[1]}")
    print(f"Final retained columns : {filtered_df.shape[1]}")
    print(f"Total features dropped  : {df.shape[1] - filtered_df.shape[1]}")

    return filtered_df, dropped_report


# =============================================================================
# EXAMPLE USAGE
# =============================================================================
# Run this on your `matrix` or raw dataframe:
filtered_matrix, filter_report = unsupervised_feature_filtering(
    df=matrix,                  # Your raw feature matrix
    missing_threshold=0.20,     # Paper threshold: max 20% missing
    variance_threshold=0.01,    # Near-zero variance filter
    corr_threshold=0.85,        # High collinearity filter
    keep_cols=['SEQN', 'frailty_label'] # Columns that should never be dropped
)
filter_report['high_missing']

[Step 1] Dropped 11 columns with > 20% missing values.
[Step 2] Dropped 0 columns with variance <= 0.01.
[Step 3] Dropped 0 redundant features with Pearson |r| > 0.85.

=== UNSUPERVISED FILTERING SUMMARY ===
Original total columns : 75
Final retained columns : 64
Total features dropped  : 11


['push/pull large [NA]',
 'little interest [NA]',
 'feeling down [NA]',
 'trouble sleeping [NA]',
 'feeling tired [NA]',
 'poor appetite [NA]',
 'feeling bad self [NA]',
 'trouble concentr [NA]',
 'urine leakage',
 'num prescriptions',
 'vitamin D']

###Multicollinearity filtering

In [ ]:
import pandas as pd
import numpy as np

# 1. Prepare your feature matrix (ensure numerical/processed data or use numeric columns)
# Drop label columns if present
cols_to_exclude = ['shrinking', 'slowness', 'weakness', 'exhaustion', 'low_activity', 'total_frailty_score', 'fried_frailty_label']
#excluded 7 variables
feat_df = filtered_matrix.drop(columns=[c for c in cols_to_exclude if c in filtered_matrix.columns])

# Keep only numeric columns for correlation calculation
numeric_feat_df = feat_df.select_dtypes(include=[np.number])

# 2. Calculate the absolute correlation matrix
corr_matrix = numeric_feat_df.corr().abs()

# 3. Create a mask to identify upper triangle correlations > 0.7
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# 4. Find features with correlation greater than 0.7
to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > 0.7)]

print(f"Columns to drop due to high correlation (> 0.7): {to_drop}")

# 5. Filter them out of your feature matrix
filtered_features_uncorrelated = numeric_feat_df.drop(columns=to_drop)

print(f"Original feature count: {numeric_feat_df.shape[1]}")
print(f"Feature count after removing multicollinearity: {filtered_features_uncorrelated.shape[1]}")
#dropped more columns because of high similarity

Columns to drop due to high correlation (> 0.7): ['house chores', 'map', 'pulse_pressure', 'RBC count', 'glycohemoglobin']
Original feature count: 64
Feature count after removing multicollinearity: 59


In [ ]:
filtered_matrix.columns = filtered_matrix.columns.str.strip()

In [ ]:
import pandas as pd
import numpy as np

def build_ffp_and_preprocess_features(filtered_matrix: pd.DataFrame):
    """
    Constructs the Fried Frailty Phenotype (FFP) labels and score from filtered_matrix,
    excludes them from the feature space, and explicitly classifies features into
    binary, float/continuous, and categorical columns.

    Parameters:
    -----------
    filtered_matrix : pd.DataFrame
        The raw merged NHANES dataset containing features, metadata, and survey columns.

    Returns:
    --------
    ffp_df : pd.DataFrame
        DataFrame containing individual FFP components, total_frailty_score, and fried_frailty_label.
    feature_df : pd.DataFrame
        The cleaned feature matrix with FFP components, scores, and labels excluded.
    binary_cols : list of str
        Columns with <= 2 unique non-null values.
    float_cols : list of str
        Continuous, ordinal, and large-cardinality numeric features cast to float64.
    categorical_cols : list of str
        Unordered categorical features with > 2 unique values.
    """
    # Ensure local reference uses filtered_matrix to prevent NameErrors
    df = filtered_matrix.copy()

    # ==========================================
    # 1. FFP Component Construction & Labeling
    # ==========================================
    component_cols = ['shrinking', 'slowness', 'weakness', 'exhaustion', 'low_activity']

    # Ensure component columns exist in the dataframe (initialize with NaN if missing)
    ffp_dict = {col: df[col] if col in df.columns else pd.Series(np.nan, index=df.index) for col in component_cols}
    ffp_df = pd.DataFrame(ffp_dict, index=df.index)

    # Calculate total positive frailty score allowing at most 1 missing component (min_count=4)
    # This evaluates to NaN if more than 1 component is missing
    ffp_df['total_frailty_score'] = ffp_df[component_cols].sum(axis=1, min_count=4)

    # Calculate final binary Fried Frailty Phenotype label:
    # Frail (1) if score >= 3, Non-frail (0) if score < 3, and NaN if score is NaN
    conditions = [
        ffp_df['total_frailty_score'] >= 3,
        ffp_df['total_frailty_score'] < 3
    ]
    choices = [1, 0]
    ffp_df['fried_frailty_label'] = np.select(conditions, choices, default=np.nan)

    # ==========================================
    # 2. Exclude Non-Feature Columns
    # ==========================================
    columns_to_exclude = component_cols + ['total_frailty_score', 'fried_frailty_label']
    drop_cols = [col for col in columns_to_exclude if col in df.columns]
    feature_df = df.drop(columns=drop_cols)

    # ==========================================
    # 3. Feature Classification & Routing
    # ==========================================
    binary_cols = []
    float_cols = []
    categorical_cols = []

    for col in feature_df.columns:
        # Handle object/string types gracefully by attempting numeric coercion
        if pd.api.types.is_string_dtype(feature_df[col]) or feature_df[col].dtype == object:
            try:
                feature_df[col] = pd.to_numeric(feature_df[col], errors='raise')
            except (ValueError, TypeError):
                # Genuinely non-numeric text columns go to categorical or string handling
                categorical_cols.append(col)
                continue

        # Analyze unique non-null values
        unique_vals = feature_df[col].dropna().unique()
        num_unique = len(unique_vals)

        # Binary Check: <= 2 unique values (typically 0 and 1)
        if num_unique <= 2:
            binary_cols.append(col)
            feature_df[col] = pd.to_numeric(feature_df[col], errors='coerce')

        else:
            # Check if it's an unordered categorical group code vs numeric/ordinal
            # (e.g., nominal categories with >2 unique text/integer codes without clear scale)
            # For this pipeline, we route continuous, ordinals, and large-range integers to float_cols
            # and explicitly cast them to float64 for standard scaling.
            float_cols.append(col)
            feature_df[col] = pd.to_numeric(feature_df[col], errors='coerce').astype('float64')

    # ==========================================
    # 4. Verification & Logging
    # ==========================================
    print(f"--- Pipeline Execution Summary ---")
    print(f"Binary columns ({len(binary_cols)}): {binary_cols}")
    print(f"Float/Ordinal columns ({len(float_cols)}): {float_cols}")
    print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")
    print(f"FFP Label Distribution:\n{ffp_df['fried_frailty_label'].value_counts(dropna=False)}")

    return ffp_df, feature_df, binary_cols, float_cols, categorical_cols

# --- Example Usage ---
# ffp_df, feature_df, binary_cols, float_cols, categorical_cols = build_ffp_and_preprocess_features(filtered_matrix)

##Fried Frailty Phenotype

In [ ]:
#Fried frailty phenotype (5 criterias)
import numpy as np
import pandas as pd

# 0. Bind the merged dataset
df = filtered_matrix.copy()

ffp_df = pd.DataFrame(index=df.index)

# 1. Exhaustion (Feeling tired / general health fallback)
if 'feeling tired [NA]' in df.columns and df['feeling tired [NA]'].notna().sum() > 0:
    ffp_df['exhaustion'] = np.where(df['feeling tired [NA]'].isna(), np.nan, (df['feeling tired [NA]'] >= 1).astype(int))
elif 'general health' in df.columns:
    ffp_df['exhaustion'] = np.where(df['general health'].isna(), np.nan, (df['general health'] >= 4).astype(int))
else:
    ffp_df['exhaustion'] = np.nan

# 2. Low Physical Activity
cond1 = df['stand long periods'] >= 2 if 'stand long periods' in df.columns else pd.Series(np.nan, index=df.index)
cond2 = df['leisure at home'] >= 2 if 'leisure at home' in df.columns else pd.Series(np.nan, index=df.index)

act_combined = np.where(cond1.fillna(False) | cond2.fillna(False), 1, 0)
act_combined = np.where(cond1.isna() & cond2.isna(), np.nan, act_combined)
ffp_df['low_activity'] = act_combined

# 3. Shrinking
if 'bmi' in df.columns:
    ffp_df['shrinking'] = np.where(df['bmi'].isna(), np.nan, (df['bmi'] <= 18.5).astype(int))
else:
    ffp_df['shrinking'] = np.nan

# 4. Slowness
if 'walk quarter mile' in df.columns:
    ffp_df['slowness'] = np.where(df['walk quarter mile'].isna(), np.nan, (df['walk quarter mile'] >= 2).astype(int))
elif 'walk up 10 steps' in df.columns:
    ffp_df['slowness'] = np.where(df['walk up 10 steps'].isna(), np.nan, (df['walk up 10 steps'] >= 2).astype(int))
else:
    ffp_df['slowness'] = np.nan

# 5. Weakness
if 'lifting/carrying' in df.columns:
    ffp_df['weakness'] = np.where(df['lifting/carrying'].isna(), np.nan, (df['lifting/carrying'] >= 2).astype(int))
else:
    ffp_df['weakness'] = np.nan

# 6. Score & Binary Label (Allowing at most 1 missing component)
component_cols = ['shrinking', 'slowness', 'weakness', 'exhaustion', 'low_activity']
ffp_df['total_frailty_score'] = ffp_df[component_cols].sum(axis=1, min_count=4)

ffp_df['fried_frailty_label'] = np.where(
    ffp_df['total_frailty_score'].isna(),
    np.nan,
    (ffp_df['total_frailty_score'] >= 3).astype(int)
)

In [ ]:
ffp_df

,exhaustion,low_activity,shrinking,slowness,weakness,total_frailty_score,fried_frailty_label
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12,1.0,1.0,0.0,1.0,0.0,3.0,1.0
13,0.0,0.0,0.0,0.0,0.0,0.0,0.0
15,0.0,1.0,0.0,0.0,0.0,1.0,0.0
23,1.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...
19746,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19748,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19749,1.0,0.0,0.0,0.0,0.0,1.0,0.0
19750,1.0,1.0,NaN,0.0,1.0,3.0,1.0


###Classification to binary, strings, floats

In [ ]:
#Could this code have the same purpose as the previous one?
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# 1. Isolate feature matrix (excluding target/label columns)
cols_to_exclude = ['shrinking', 'slowness', 'weakness', 'exhaustion', 'low_activity', 'total_frailty_score', 'fried_frailty_label']
feature_df = filtered_matrix.drop(columns=[c for c in cols_to_exclude if c in filtered_matrix.columns])
feature_df.columns = feature_df.columns.str.strip()

binary_cols = []
float_cols = []
string_cols = []
remaining_integer_cols = [] #why did we create more?

# 2. Apply your primary sorting rules
for col in feature_df.columns:
    series = feature_df[col]

    # Rule 3: String columns - All columns that have words or letters -> OneHotEncoder
    if series.dtype == 'object' or pd.api.types.is_string_dtype(series):
        string_cols.append(col)
        continue

    num_series = pd.to_numeric(series, errors='coerce')
    clean_series = num_series.dropna()

    if len(clean_series) == 0:
        continue

    # Check if numbers have decimals / are not counting numbers
    is_float = not np.all(clean_series == clean_series.astype(int))

    if is_float:
        # Rule 2: Floating point numbers -> StandardScaler list
        float_cols.append(col)
    else:
        unique_vals = sorted(clean_series.unique())
        # Rule 1: Binary values - column is only 0s or 1s -> keep as they are (passthrough)
        if set(unique_vals).issubset({0, 1}):
            binary_cols.append(col)
        else:
            # Remaining counting numbers (start at 0/1 and go beyond 1)
            remaining_integer_cols.append(col)


# =============================================================================
# 3. SPLIT REMAINING INTEGER COLUMNS INTO ORDINAL VS CATEGORICAL
# =============================================================================
ordinal_cols = []
categorical_cols = []

# Heuristic inspection for remaining counting numbers:
# - Ordinal: Ordered scales (e.g., scale steps like 0 to 4) -> StandardScaler
# - Categorical: Unordered category codes (e.g., group IDs) -> OneHotEncoder
for col in remaining_integer_cols:
    unique_count = feature_df[col].nunique()
    max_val = feature_df[col].max()

    # Small discrete steps are typically ordinal questionnaire scales
    if unique_count <= 6 and max_val <= 5:
        ordinal_cols.append(col)
    else:
        categorical_cols.append(col)

print("=== FINAL COLUMN GROUPINGS ===")
print(f"Binary (Passthrough) [{len(binary_cols)}]")
print(f"Floating-point (StandardScaler) [{len(float_cols)}]")
print(f"String (OneHotEncoder) [{len(string_cols)}]")
print(f"Ordinal (StandardScaler) [{len(ordinal_cols)}]")
print(f"Categorical (OneHotEncoder) [{len(categorical_cols)}]")


# =============================================================================
# 4. BUILD THE PREPROCESSING PIPELINE
# =============================================================================
float_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('bin', 'passthrough', binary_cols),
        ('float', float_transformer, float_cols),
        ('ord', ordinal_transformer, ordinal_cols),
        ('cat', categorical_transformer, categorical_cols + string_cols)
    ],
    remainder='drop'
)

# Execute transformation pipeline
X_final_processed = preprocessor.fit_transform(feature_df)

print("\n=== PIPELINE STATUS ===")
print("Processed feature matrix shape:", X_final_processed.shape)

=== FINAL COLUMN GROUPINGS ===
Binary (Passthrough) [14]
Floating-point (StandardScaler) [28]
String (OneHotEncoder) [0]
Ordinal (StandardScaler) [17]
Categorical (OneHotEncoder) [5]

=== PIPELINE STATUS ===
Processed feature matrix shape: (3487, 507)


## Supervised feature selection

### LASSO

In [ ]:
import pandas as pd
import numpy as np

# 1. Extract y (the frailty label) from ffp_df
y = ffp_df['fried_frailty_label']

# 2. Ensure X_final_processed is a DataFrame (handling it just in case it's a numpy array)
if isinstance(X_final_processed, np.ndarray):
    cols = feature_names_list if 'feature_names_list' in locals() else [f"feat_{i}" for i in range(X_final_processed.shape[1])]
    X_df = pd.DataFrame(X_final_processed, columns=cols, index=ffp_df.index)
else:
    X_df = X_final_processed.copy()

# 3. Safely drop demographic anchors from the candidate features
X_candidates = X_df.drop(columns=["age", "sex"], errors="ignore")

# 4. Build the valid mask and clean both matrices together using matched indices
valid_mask = y.notna() & X_candidates.notna().all(axis=1)
X_clean = X_candidates.loc[valid_mask]
y_clean = y.loc[valid_mask]

print(f"Ready for LASSO! Evaluated samples: {len(y_clean)}, Features: {X_clean.shape[1]}")

Ready for LASSO! Evaluated samples: 3251, Features: 507


In [45]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

def perform_lasso_fixed_selection(X, y, c_value=0.1):
    # 0. Safety conversions
    if isinstance(X, np.ndarray):
        cols = [f"feat_{i}" for i in range(X.shape[1])]
        X = pd.DataFrame(X, columns=cols)
    elif not isinstance(X, pd.DataFrame):
        X = pd.DataFrame(X)

    if isinstance(y, np.ndarray):
        y = pd.Series(y, index=X.index)

    # 1. Complete Case Filter
    valid_mask = y.notna() & X.notna().all(axis=1)
    X_clean = X.loc[valid_mask]
    y_clean = y.loc[valid_mask]

    # 2. Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_clean)

    # 3. Fit standard L1 Logistic Regression with a controlled C parameter
    # (Higher C = less penalty, meaning more features are allowed to survive)
    lasso_model = LogisticRegression(
        penalty='l1',
        solver='liblinear',
        C=c_value,
        max_iter=5000,
        random_state=42
    )

    lasso_model.fit(X_scaled, y_clean)

    # 4. Extract weights
    raw_coefficients = lasso_model.coef_[0]
    absolute_coefficients = np.abs(raw_coefficients)

    feature_df = pd.DataFrame({
        'feature': X_clean.columns,
        'absolute_coefficient': absolute_coefficients,
        'raw_coefficient': raw_coefficients
    })

    selected_features_df = feature_df[feature_df['absolute_coefficient'] > 0].copy()
    selected_features_df = selected_features_df.sort_values(by='absolute_coefficient', ascending=False)

    # 5. Summary
    print(f"=== LASSO Execution Summary (C={c_value}) ===")
    print(f"Total input features evaluated: {X_clean.shape[1]}")
    print(f"Count of selected non-zero variables: {len(selected_features_df)}")
    print("\n--- Top Selected Variables ---")

    if not selected_features_df.empty:
        print(selected_features_df.head(25).to_string(index=False))
    else:
        print("Still 0 features. Try increasing C (e.g., C=1.0).")

    return lasso_model, selected_features_df

# ==========================================
# Run it here (Adjust C if needed: try 0.05, 0.1, or 1.0)
# ==========================================
lasso_model, lasso_features_df = perform_lasso_fixed_selection(
    X_final_processed,
    ffp_df['fried_frailty_label'],
    c_value=0.1
)

=== LASSO Execution Summary (C=0.1) ===
Total input features evaluated: 507
Count of selected non-zero variables: 58

--- Top Selected Variables ---
 feature  absolute_coefficient  raw_coefficient
feat_178              0.178091         0.178091
feat_344              0.155922         0.155922
feat_497              0.142330         0.142330
  feat_8              0.136154        -0.136154
feat_134              0.135790         0.135790
feat_357              0.130609         0.130609
feat_330              0.114100         0.114100
 feat_76              0.110477         0.110477
 feat_59              0.107042        -0.107042
feat_345              0.103107         0.103107
 feat_61              0.101067         0.101067
feat_207              0.096564         0.096564
feat_422              0.094995         0.094995
feat_388              0.091532         0.091532
feat_171              0.088118         0.088118
 feat_50              0.087923        -0.087923
feat_212              0.085649     

In [48]:
# Run with C set precisely to hit 23 features
lasso_model, lasso_features_df = perform_lasso_fixed_selection(
    X_final_processed,
    ffp_df['fried_frailty_label'],
    c_value=0.0585  # Right between 0.0568 and 0.0614
)

print(f"\n✨ Final selected features count: {len(lasso_features_df)}")

=== LASSO Execution Summary (C=0.0585) ===
Total input features evaluated: 507
Count of selected non-zero variables: 24

--- Top Selected Variables ---
 feature  absolute_coefficient  raw_coefficient
feat_178              0.118668         0.118668
feat_344              0.079215         0.079215
feat_134              0.076946         0.076946
feat_357              0.066150         0.066150
feat_345              0.055630         0.055630
feat_330              0.050278         0.050278
feat_497              0.050217         0.050217
feat_422              0.038218         0.038218
feat_207              0.038217         0.038217
feat_444              0.037510         0.037510
feat_437              0.033549         0.033549
feat_413              0.033537         0.033537
feat_212              0.033537         0.033537
feat_418              0.033536         0.033536
 feat_76              0.031242         0.031242
feat_218              0.024357         0.024357
  feat_8              0.010481  

###VSURF

In [52]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

def perform_tuned_strict_vsurf(X, y):
    # 0. Safety conversions for DataFrame format
    if isinstance(X, np.ndarray):
        cols = [f"feat_{i}" for i in range(X.shape[1])]
        X = pd.DataFrame(X, columns=cols)
    elif not isinstance(X, pd.DataFrame):
        X = pd.DataFrame(X)

    if isinstance(y, np.ndarray):
        y = pd.Series(y, index=X.index)

    # 1. Complete Case Filtering
    valid_mask = y.notna() & X.notna().all(axis=1)
    X_clean = X.loc[valid_mask]
    y_clean = y.loc[valid_mask]

    print("=== Running Tuned Autonomous VSURF Selection ===")
    print(f"Total initial features evaluated: {X_clean.shape[1]}")

    # Step 1: Initial Random Forest Ranking
    rf_base = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight='balanced')
    rf_base.fit(X_clean, y_clean)

    importances = rf_base.feature_importances_
    sorted_indices = np.argsort(importances)[::-1]
    sorted_features = X_clean.columns[sorted_indices]

    # Step 2: Initial OOB Error curve to find the ceiling
    oob_errors = []
    max_test = min(100, len(sorted_features))

    for k in range(1, max_test + 1):
        feats = sorted_features[:k]
        rf_sub = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=42, n_jobs=-1, class_weight='balanced')
        rf_sub.fit(X_clean[feats], y_clean)
        oob_errors.append(1.0 - rf_sub.oob_score_)

    oob_errors = np.array(oob_errors)
    min_idx = np.argmin(oob_errors)

    # Start step-down from the minimum error point
    best_features = list(sorted_features[:min_idx + 1])

    rf_curr = RandomForestClassifier(n_estimators=200, oob_score=True, random_state=42, n_jobs=-1, class_weight='balanced')
    rf_curr.fit(X_clean[best_features], y_clean)
    best_oob_err = 1.0 - rf_curr.oob_score_

    # Step 3: Aggressive Backward Pruning
    # Stricter rule: We drop features unless keeping them strictly lowers the error.
    print("Executing aggressive step-down redundancy pruning...")
    reduced = True
    while reduced and len(best_features) > 1:
        reduced = False
        # Test dropping the bottom 2 least important features at once to escape flat plateaus
        drop_count = min(2, len(best_features) - 1)
        candidate_features = best_features[:-drop_count]

        rf_test = RandomForestClassifier(n_estimators=200, oob_score=True, random_state=42, n_jobs=-1, class_weight='balanced')
        rf_test.fit(X_clean[candidate_features], y_clean)
        test_oob_err = 1.0 - rf_test.oob_score_

        # If dropping them doesn't significantly hurt accuracy, accept the drop
        if test_oob_err <= best_oob_err + 0.001:
            best_features = candidate_features
            best_oob_err = test_oob_err
            reduced = True

    final_df = pd.DataFrame({
        'feature': best_features,
        'importance': [importances[list(X_clean.columns).index(f)] for f in best_features]
    }).sort_values(by='importance', ascending=False).reset_index(drop=True)

    print(f"\n--- Tuned Autonomous VSURF Selection Complete ---")
    print(f"VSURF naturally selected: {len(final_df)} features")
    print("\nTop Selected Features:")
    print(final_df.head(15).to_string(index=False))

    return rf_base, final_df

# ==========================================
# Run the tuned version:
# ==========================================
vsurf_model, vsurf_features_df = perform_tuned_strict_vsurf(
    X_final_processed,
    ffp_df['fried_frailty_label']
)

=== Running Tuned Autonomous VSURF Selection ===
Total initial features evaluated: 507
Executing aggressive step-down redundancy pruning...

--- Tuned Autonomous VSURF Selection Complete ---
VSURF naturally selected: 53 features

Top Selected Features:
feature  importance
feat_40    0.031257
feat_15    0.029686
feat_14    0.029533
feat_17    0.026827
feat_39    0.026199
feat_41    0.026169
feat_18    0.026008
feat_35    0.024560
feat_16    0.024446
feat_27    0.024046
feat_19    0.024024
feat_24    0.023977
feat_23    0.023536
feat_31    0.022824
feat_29    0.022600


###Boruta